In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
# Lecture Bronze
# Les id sont changé dans la premier phase de Amen
incidents = spark.table("workspace.final_project.bronze_incidents")
incidents_quarantine = incidents.limit(0).withColumn("reason", lit(''))
display(incidents)
# Il faudra prendre les silvers quand ils seront prets.
suppliers = spark.table("workspace.final_project.bronze_suppliers")
orders = spark.table("workspace.final_project.bronze_orders")


In [0]:
# incident_type: normalisation string
incidents = incidents.withColumn(
    "incident_type",
    initcap(
        regexp_replace(
            regexp_replace(trim(col("incident_type")), "[^a-zA-Z0-9 ]", ""),
            "[àáâä]", "a"
        )
    )
)

In [0]:
# severity: normalisation string
incidents = incidents.withColumn(
    "severity",
    initcap(
        regexp_replace(trim(col("severity")), "[^a-zA-Z0-9 ]", "")
    )
)

In [0]:
# incident_date: normalisation date + quanrantaine des nulls
# Ne pas supprimer les date a null: laisser null ou remplacer par la date d'arriver ?
incidents = incidents.withColumn(
    "incident_date_parsed",
    coalesce(
        to_date("incident_date", "yyyy-MM-dd"),
        to_date("incident_date", "dd/MM/yyyy"),
        to_date("incident_date", "yyyy/MM/dd")
    )
)

# lignes invalides
invalid_date = (
    incidents
    .filter(col("incident_date_parsed").isNull())
    .withColumn("reason", lit("Invalid incident_date format"))
    .drop("incident_date_parsed")
)
incidents_quarantine = incidents_quarantine.unionByName(invalid_date)

# lignes valides
incidents = (
    incidents
    .filter(col("incident_date_parsed").isNotNull())
    .drop("incident_date")
    .withColumnRenamed("incident_date_parsed", "incident_date")
)

In [0]:
# supplier_id: normalisation int + quanrantaine des nulls 
# + vérification présence dans la table bronze_suppliers
# Si on ne trouve pas supplier_id on va le chercher dans la table orders
# Il faut traiter order_id avant ducoup

incidents = incidents.withColumn(
    "supplier_id_parsed",
    col("supplier_id").try_cast(LongType())
)

# lignes invalides (format incorrect)
invalid_supplier_format = (
    incidents
    .filter(col("supplier_id_parsed").isNull())
    .withColumn("reason", lit("Invalid supplier_id format"))
    .drop("supplier_id_parsed")
)

incidents_quarantine = incidents_quarantine.unionByName(invalid_supplier_format)

# lignes valides
incidents = (
    incidents
    .filter(col("supplier_id_parsed").isNotNull())
    .drop("supplier_id")
    .withColumnRenamed("supplier_id_parsed", "supplier_id")
)

# supplier_id absent de la table suppliers
invalid_supplier_not_found = (
    incidents
    .join(
        suppliers.select("supplier_id"),
        on="supplier_id",
        how="left_anti"
    )
    .withColumn("reason", lit("supplier_id not found in suppliers table"))
)

incidents_quarantine = incidents_quarantine.unionByName(invalid_supplier_not_found)

# garder uniquement les supplier_id présents dans suppliers
incidents = incidents.join(
    suppliers.select("supplier_id"),
    on="supplier_id",
    how="inner"
)

In [0]:
# order_id: normalisation int + quanrantaine des nulls 
# + vérification présence dans la table bronze_orders

incidents = incidents.withColumn(
    "order_id_parsed",
    col("order_id").try_cast(LongType())
)

# lignes invalides (format incorrect)
invalid_order_format = (
    incidents
    .filter(col("order_id_parsed").isNull())
    .withColumn("reason", lit("Invalid order_id format"))
    .drop("order_id_parsed")
)

incidents_quarantine = incidents_quarantine.unionByName(invalid_order_format)

# lignes valides
incidents = (
    incidents
    .filter(col("order_id_parsed").isNotNull())
    .drop("order_id")
    .withColumnRenamed("order_id_parsed", "order_id")
)


# order_id absent de la table orders
invalid_order_not_found = (
    incidents
    .join(
        orders.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .withColumn("reason", lit("order_id not found in orders table"))
)

incidents_quarantine = incidents_quarantine.unionByName(invalid_order_not_found)

# garder uniquement les order_id présents dans orders
incidents = incidents.join(
    orders.select("order_id"),
    on="order_id",
    how="inner"
)

In [0]:
incidents = incidents.withColumn(
    "description",
    trim(col("description"))
)

In [0]:
incidents.show()

In [0]:
incidents.write.mode("overwrite").saveAsTable(
    "workspace.final_project.silver_incidents"
)
incidents_quarantine.write.mode("overwrite").saveAsTable(
    "workspace.final_project.incidents_quarantine"
)